# Set 10 – Support Vector Regression (SVR)

Die SVR sucht eine möglichst flache Funktion mit **$\varepsilon$-Tunnel**. Abweichungen innerhalb des Tunnels werden nicht bestraft.

## Lernziele

- `SVR` in einer Pipeline einsetzen
- linear, polynomial und RBF vergleichen
- `C`, `gamma` und `epsilon` unterscheiden
- Tunnel und Support-Vektoren visualisieren
- MAE, RMSE, $R^2$ und Cross-Validation verwenden


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
RANDOM_STATE=42
plt.style.use("seaborn-v0_8-whitegrid")
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Kontrollierte Daten

Eine verrauschte Sinusbeziehung mit einem Merkmal macht Modellkurve, Tunnel und Support-Vektoren sichtbar.


In [ ]:
rng=np.random.default_rng(RANDOM_STATE); X=np.sort(rng.uniform(0,8,280)).reshape(-1,1)
y=np.sin(X[:,0])+.18*X[:,0]+rng.normal(0,.22,len(X))
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=RANDOM_STATE)
plt.figure(figsize=(8,5)); plt.scatter(X_train[:,0],y_train,s=24,alpha=.65,label="Training"); plt.scatter(X_test[:,0],y_test,s=28,label="Test"); plt.legend(); plt.show()


## 2. `SVR`: Parameter

| Name | Bedeutung |
|---|---|
| `kernel` | `linear`, `poly`, `rbf`, `sigmoid` |
| `C` | Strafe für Abweichungen außerhalb des Tunnels |
| `epsilon` | halbe Tunnelbreite; Fehler darin werden ignoriert |
| `gamma` | Reichweite bei RBF/Poly: klein = glatt, groß = lokal |
| `degree` | Grad bei `poly` |
| `fit`, `predict` | lernen bzw. kontinuierliche Werte vorhersagen |

Größeres `C` passt Fehler stärker an; größeres `epsilon` führt oft zu weniger Support-Vektoren; großes `gamma` kann die Kurve unruhig machen.


In [ ]:
model=Pipeline([("scaler",StandardScaler()),("svr",SVR(kernel="rbf",C=10,gamma="scale",epsilon=.15))]).fit(X_train,y_train)
def metrics(a,p): return {"MAE":mean_absolute_error(a,p),"RMSE":np.sqrt(mean_squared_error(a,p)),"R2":r2_score(a,p)}
print({k:round(v,3) for k,v in metrics(y_test,model.predict(X_test)).items()})


## 3. $\varepsilon$-Tunnel und Support-Vektoren

`support_` enthält die Trainingsindizes entscheidender Punkte. Punkte außerhalb oder am Tunnelrand können Support-Vektoren werden.


In [ ]:
def plot_svr(model,X,y,title,ax=None):
 if ax is None: _,ax=plt.subplots(figsize=(8,5))
 grid=np.linspace(X.min(),X.max(),500).reshape(-1,1); p=model.predict(grid); eps=model.named_steps["svr"].epsilon; idx=model.named_steps["svr"].support_
 ax.scatter(X[:,0],y,s=22,alpha=.5,label="Training"); ax.scatter(X[idx,0],y[idx],s=70,facecolors="none",edgecolors="black",label="Support")
 ax.plot(grid[:,0],p,color="crimson",lw=2); ax.fill_between(grid[:,0],p-eps,p+eps,color="crimson",alpha=.14,label=f"epsilon={eps}")
 ax.set(title=title,xlabel="x",ylabel="y"); ax.legend()
plot_svr(model,X_train,y_train,"RBF-SVR und epsilon-Tunnel"); plt.show(); print("Support:",model.named_steps["svr"].support_.size)


## 4. Kernels vergleichen

Linear kann nur eine Gerade bilden. Poly bildet polynomial gekrümmte Beziehungen ab. RBF ist lokal und flexibel.


In [ ]:
ests={"Linear":SVR(kernel="linear",C=10,epsilon=.15),"Poly":SVR(kernel="poly",degree=3,C=10,epsilon=.15),"RBF":SVR(kernel="rbf",C=10,epsilon=.15)}
fig,axes=plt.subplots(1,3,figsize=(18,4.8)); rows=[]
for ax,(name,est) in zip(axes,ests.items()):
 pipe=Pipeline([("scaler",StandardScaler()),("svr",est)]).fit(X_train,y_train)
 rows.append({"Kernel":name,**metrics(y_test,pipe.predict(X_test)),"Support":pipe.named_steps["svr"].support_.size}); plot_svr(pipe,X_train,y_train,name,ax)
plt.tight_layout(); plt.show(); display(pd.DataFrame(rows).round(3))


## 5. `C`, `gamma`, `epsilon`

Beobachte Glätte, Tunnelbreite und Zahl der markierten Punkte. Der breite Tunnel toleriert größere Abweichungen.


In [ ]:
settings=[(.3,.3,.15),(30,.3,.15),(30,3,.15),(30,.3,.45)]; fig,axes=plt.subplots(2,2,figsize=(13,9))
for ax,(C_value,gamma_value,eps) in zip(axes.ravel(),settings):
 pipe=Pipeline([("scaler",StandardScaler()),("svr",SVR(kernel="rbf",C=C_value,gamma=gamma_value,epsilon=eps))]).fit(X_train,y_train)
 plot_svr(pipe,X_train,y_train,f"C={C_value}, gamma={gamma_value}, epsilon={eps}",ax)
plt.tight_layout(); plt.show()


## 6. Hyperparameter-Suche

scikit-learn maximiert Scores, daher ist RMSE negativ gespeichert: näher an null ist besser.


In [ ]:
pipe=Pipeline([("scaler",StandardScaler()),("svr",SVR())])
grid=[{"svr__kernel":["linear"],"svr__C":[.1,1,10],"svr__epsilon":[.05,.2,.5]},
{"svr__kernel":["poly"],"svr__C":[1,10],"svr__gamma":[.1,1],"svr__degree":[2,3],"svr__epsilon":[.05,.2]},
{"svr__kernel":["rbf"],"svr__C":[1,10,100],"svr__gamma":[.03,.1,1],"svr__epsilon":[.05,.2,.5]}]
search=GridSearchCV(pipe,grid,scoring="neg_root_mean_squared_error",cv=5,n_jobs=-1,return_train_score=True).fit(X_train,y_train)
print("Beste Parameter:",search.best_params_); print("CV-RMSE:",round(-search.best_score_,3))
res=pd.DataFrame(search.cv_results_); display(res[["params","mean_train_score","mean_test_score"]].sort_values("mean_test_score",ascending=False).head(10).round(3))


In [ ]:
best=search.best_estimator_; p=best.predict(X_test); print({k:round(v,3) for k,v in metrics(y_test,p).items()})
plot_svr(best,X_train,y_train,"Bestes Modell"); plt.show()
plt.figure(figsize=(6,6)); plt.scatter(y_test,p); lim=[min(y_test.min(),p.min()),max(y_test.max(),p.max())]; plt.plot(lim,lim,"--",color="black"); plt.xlabel("Tatsächlich"); plt.ylabel("Vorhergesagt"); plt.show()


## Fazit

`epsilon` legt die straffreie Zone fest, `C` die Strafe außerhalb und `gamma` die Reichweite. Pipeline, CV und ein unangetasteter Testdatensatz bleiben zentral.
